In [1]:
print("T5 model training notebook started")

T5 model training notebook started


In [2]:
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/validation.csv")
test_df = pd.read_csv("../data/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (551, 3)
Validation: (69, 3)
Test: (69, 3)


In [20]:
train_df["input_text"] = "question: " + train_df["Question"].astype(str)
train_df["target_text"] = train_df["Answer"].astype(str)

val_df["input_text"] = "question: " + val_df["Question"].astype(str)
val_df["target_text"] = val_df["Answer"].astype(str)

test_df["input_text"] = "question: " + test_df["Question"].astype(str)
test_df["target_text"] = test_df["Answer"].astype(str)

train_df[["input_text", "target_text"]].head()

,input_text,target_text
0,question: what does memantine look like,"Color - ORANGE, Shape - CAPSULE (biconvex), Sc..."
1,question: how does a 20 mcg bedford norton tra...,25 mcg fentanyl/hr = 40 mg oral / 20 mg IV oxy...
2,question: what mg norco comes in,... NORCO® 5/325 ... NORCO® 7.5/325 ... NORCO®...
3,question: vyvanse 10 what is all in this pill ...,The following adverse reactions are discussed ...
4,question: dtap/tdap/td vaccines how often is t...,"Routine Vaccination of Infants and Children, A..."


In [5]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "target_text"]])

train_dataset

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 551
})

In [6]:
model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Tokenizer and model loaded successfully")

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\Serhat\Desktop\medicine-chatbot-nlp\medicine-chatbot-nlp\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Serhat\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizer and model loaded successfully


In [7]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [8]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

print("Tokenization completed")

Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Tokenization completed


In [9]:
training_args = TrainingArguments(
    output_dir="../models/t5-small-medicationqa",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="../models/logs",
    logging_steps=50,
    save_total_limit=1
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

print("Trainer created successfully")

Trainer created successfully


In [12]:
trainer.train()

c:\Users\Serhat\Desktop\medicine-chatbot-nlp\medicine-chatbot-nlp\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.505005,1.324668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=276, training_loss=1.6913791324781335, metrics={'train_runtime': 146.7315, 'train_samples_per_second': 3.755, 'train_steps_per_second': 1.881, 'total_flos': 18643333152768.0, 'train_loss': 1.6913791324781335, 'epoch': 1.0})

In [13]:
def generate_answer(question):

    input_text = "question: " + question

    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids

    outputs = model.generate(
        input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [14]:
question = "Can I take this medicine with food?"

response = generate_answer(question)

print("QUESTION:")
print(question)

print("\nMODEL ANSWER:")
print(response)

QUESTION:
Can I take this medicine with food?

MODEL ANSWER:
Do not take this medicine with food. Do not take this medicine with food.


In [15]:
question = "What should I do if I miss a dose?"

response = generate_answer(question)

print(response)

In [16]:
question = "What is rivastigmine used for?"

response = generate_answer(question)

print(response)

rivastigmine is used to treat a variety of bacterial infections. It is used to treat a variety of bacterial infections. It is used to treat a variety of bacterial infections.


In [21]:
model.save_pretrained("../models/t5-small-medicationqa-final")
tokenizer.save_pretrained("../models/t5-small-medicationqa-final")

print("Model and tokenizer saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully.


In [22]:
import os

os.listdir("../models/t5-small-medicationqa-final")

['config.json',
 'generation_config.json',
 'model.safetensors',
 'tokenizer.json',
 'tokenizer_config.json']